# Razonamiento causal: correlación no es causa

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

**Facsímil 12 · Lógica, conocimiento e incertidumbre** — capítulo 5 (razonamiento causal).

«Donde se venden más helados se ahoga más gente.» El dato es cierto, pero sería absurdo prohibir los
helados para salvar vidas: hay una causa oculta, el **calor**, que dispara las dos cosas a la vez. Esa
trampa —confundir lo que va **junto** con lo que **provoca**— está en el corazón de medio mundo de
decisiones equivocadas. Este cuaderno te da las herramientas para no caer: el confusor, la paradoja de
Simpson, el operador **do** de Pearl y la **fórmula de ajuste**, todo construido a mano con numpy, sin
librerías de caja negra.

### Qué vas a aprender
- Por qué dos cosas pueden estar **correlacionadas sin que una cause la otra**: el papel del **confusor**.
- La **paradoja de Simpson**: una tendencia que se **invierte** al mirar los subgrupos.
- El **operador do**: la diferencia entre **observar** $P(Y\mid X)$ e **intervenir** $P(Y\mid \text{do}(X))$.
- La **fórmula de ajuste por puerta trasera**, que recupera el efecto causal real a partir de datos de observación.
- El **colisionador**: cómo condicionar sobre un efecto común **inventa** una correlación que no existía.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.
Todas las cifras de los textos salen de ejecutar el código; si cambias un dato, cambiarán.

### Cuánto cuesta
Unos 25 minutos. CPU; sin GPU ni claves. Solo numpy y matplotlib (ya vienen en Colab).

In [ ]:
# Colab ya los trae. En tu maquina:  pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("Listo.")

## 1. Helados y ahogamientos: una correlación espuria

Vamos a fabricar el clásico. Una variable oculta, la **temperatura**, es la causa común: cuando aprieta
el calor se venden más helados **y** se baña más gente (y por desgracia hay más ahogamientos). Los
helados y los ahogamientos **no** se influyen entre sí; sólo comparten una causa. Aun así, si miramos
sólo esas dos columnas, irán de la mano.

El diagrama causal real es:

```
        Temperatura  (el confusor: la causa comun)
          /        \
         v          v
     Helados    Ahogamientos     <- NO hay flecha entre ellos
```

In [ ]:
n = 500
temp = rng.uniform(10, 35, n)                      # temperatura en grados
helados      = 50 + 8.0 * temp + rng.normal(0, 30, n)   # ventas de helados
ahogamientos = 1  + 0.40 * temp + rng.normal(0, 2.0, n) # numero de ahogamientos
# Helados y ahogamientos NO se usan el uno al otro: solo dependen de temp.

def corr(a, b):
    return float(np.corrcoef(a, b)[0, 1])

r_he = corr(helados, ahogamientos)
print(f"Correlacion helados <-> ahogamientos: {r_he:.3f}")
print("Alta y positiva... pero ninguno causa al otro.")

La correlación es fuerte y positiva, y sin embargo sabemos —porque hemos fabricado los datos— que un
helado no ahoga a nadie. Es una **correlación espuria**: nace entera del confusor. Para verlo, pintamos
la nube de puntos y **coloreamos cada punto según la temperatura**.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2))
sc = ax.scatter(helados, ahogamientos, c=temp, cmap="coolwarm", s=18, edgecolor="none")
ax.set_xlabel("ventas de helados")
ax.set_ylabel("ahogamientos")
ax.set_title("Correlacion espuria: el color (temperatura) lo explica todo")
plt.colorbar(sc, label="temperatura")
plt.tight_layout(); plt.show()
print("Los puntos rojos (calor) se agolpan arriba a la derecha; los azules (frio), abajo a la izquierda.")

### Controlar el confusor: la correlación parcial

¿Cómo comprobamos que la relación es falsa? **Quitando** el efecto de la temperatura. Hacemos una
regresión de cada variable sobre la temperatura, nos quedamos con los **residuos** (lo que queda sin
explicar por el calor) y miramos si los residuos siguen correlacionados. Si la relación era sólo cosa
del confusor, la correlación de los residuos debe **caer cerca de cero**.

In [ ]:
def residuos(y, x):
    'Residuos de y tras restarle su mejor recta sobre x (minimos cuadrados).'
    A = np.vstack([x, np.ones_like(x)]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return y - A @ coef

res_hel = residuos(helados, temp)
res_aho = residuos(ahogamientos, temp)
r_parcial = corr(res_hel, res_aho)

print(f"Correlacion bruta        (helados, ahogamientos) = {r_he:.3f}")
print(f"Correlacion parcial dado temp                    = {r_parcial:.3f}")
print("\nAl controlar la temperatura, la correlacion se desploma: era todo del confusor.")

## 2. La paradoja de Simpson: la tendencia que se da la vuelta

Lo anterior engaña, pero al menos no cambia de signo. La **paradoja de Simpson** es peor: una tendencia
que apunta en un sentido **se invierte** al separar por el confusor. Usamos un caso real y famoso, un
estudio de 1986 sobre dos tratamientos para cálculos renales (Charig y otros). Llamemos a los
tratamientos **A** y **B**, y al confusor el **tamaño del cálculo** (pequeño o grande), que mide la
gravedad.

In [ ]:
# Datos reales (Charig et al., 1986): (exitos, total) por tratamiento y tamano.
datos = {
    ("A", "pequeno"): (81, 87),
    ("A", "grande"):  (192, 263),
    ("B", "pequeno"): (234, 270),
    ("B", "grande"):  (55, 80),
}

def tasa(exitos, total):
    return exitos / total

print("Tasa de exito por subgrupo:")
for (trat, tam), (e, t) in datos.items():
    print(f"  Tratamiento {trat}, calculo {tam:8s}: {e:3d}/{t:3d} = {tasa(e,t)*100:5.1f}%")

In [ ]:
# Agregado: juntamos los dos tamanos de cada tratamiento.
agg = {}
for trat in ["A", "B"]:
    e = sum(datos[(trat, tam)][0] for tam in ["pequeno", "grande"])
    t = sum(datos[(trat, tam)][1] for tam in ["pequeno", "grande"])
    agg[trat] = (e, t)
    print(f"Tratamiento {trat} en TOTAL: {e}/{t} = {tasa(e,t)*100:.1f}%")

print()
print(f"En calculos pequenos gana: {'A' if tasa(*datos[('A','pequeno')])>tasa(*datos[('B','pequeno')]) else 'B'}")
print(f"En calculos grandes  gana: {'A' if tasa(*datos[('A','grande')]) >tasa(*datos[('B','grande')])  else 'B'}")
print(f"En el TOTAL           gana: {'A' if tasa(*agg['A'])>tasa(*agg['B']) else 'B'}")

Léelo despacio, porque es para frotarse los ojos: el tratamiento **A** es mejor en los cálculos
pequeños **y** mejor en los grandes, pero **peor en el total**. ¿Cómo puede ser? Por el confusor: a los
casos graves (cálculos grandes, más difíciles de curar) se les dio sobre todo el tratamiento **A**, y a
los leves sobre todo el **B**. El total de **A** está lastrado por estar lleno de casos difíciles. La
media agregada **mezcla** poblaciones distintas y miente.

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.2))
tam_orden = ["pequeno", "grande", "TOTAL"]
x = np.arange(len(tam_orden))
w = 0.36

def serie(trat):
    vals = [tasa(*datos[(trat, "pequeno")]), tasa(*datos[(trat, "grande")]), tasa(*agg[trat])]
    return [v * 100 for v in vals]

ax.bar(x - w/2, serie("A"), w, label="Tratamiento A", color="#444444")
ax.bar(x + w/2, serie("B"), w, label="Tratamiento B", color="#bbbbbb")
ax.set_xticks(x); ax.set_xticklabels(["calculo pequeno", "calculo grande", "TOTAL (mezcla)"])
ax.set_ylabel("tasa de exito (%)"); ax.set_ylim(0, 100)
ax.set_title("Paradoja de Simpson: A gana en cada grupo, pierde en el total")
ax.legend()
plt.tight_layout(); plt.show()

La moraleja: **agregar sin pensar en el confusor puede invertir la conclusión**. ¿Cuál es entonces el
mejor tratamiento? El que gana **dentro de cada subgrupo comparable**: el A. Para decidir bien hay que
comparar manzanas con manzanas, no una mezcla con otra.

## 3. Un modelo causal de juguete y el operador `do`

Para razonar con precisión necesitamos un **modelo causal**: un grafo con sus tablas de probabilidad.
Tomamos tres variables binarias (sí/no = 1/0):

- **Z**, el confusor (por ejemplo, *paciente de edad avanzada*).
- **X**, la acción o tratamiento.
- **Y**, el resultado (por ejemplo, *recuperación*).

con esta estructura, donde Z empuja **a la vez** a X y a Y:

```
        Z  (confusor)
       / \
      v   v
      X-->Y
```

Z influye en quién recibe X (los mayores se tratan más) y también, por su cuenta, en Y. Esa flecha
`Z -> X` es justo la **puerta trasera** que ensucia la comparación. Definimos las tablas con números
cómodos.

In [ ]:
P_Z1 = 0.5                       # P(Z = si)
P_X1_dado_Z = {1: 0.8, 0: 0.2}   # P(X = si | Z): los Z=si reciben X mucho mas

# P(Y = si | X, Z)  ->  clave (X, Z)
P_Y1 = {
    (0, 0): 0.20,   # sin X, sin Z
    (1, 0): 0.40,   # con X, sin Z      (X aporta +0.20)
    (0, 1): 0.60,   # sin X, con Z
    (1, 1): 0.80,   # con X, con Z      (X aporta +0.20)
}
# Fijate: el efecto VERDADERO de X sobre Y, fijado Z, es +0.20 en ambos niveles.
print("Tablas listas. Efecto causal real de X (a Z fijo): +0.20 en cada estrato.")

### Observar: $P(Y\mid X)$

Primero hacemos lo que haría un analista ingenuo: mirar los datos tal cual y comparar la tasa de
recuperación de quienes recibieron X frente a quienes no. Eso es $P(Y\mid X=1) - P(Y\mid X=0)$. Lo
calculamos a partir de la conjunta del modelo.

In [ ]:
def p_z(z):
    return P_Z1 if z else 1 - P_Z1

def p_x_dado_z(x, z):
    px1 = P_X1_dado_Z[z]
    return px1 if x else 1 - px1

def p_y_dado_xz(y, x, z):
    py1 = P_Y1[(x, z)]
    return py1 if y else 1 - py1

def p_observando_Y1_dado_X(x):
    'P(Y=1 | X=x) OBSERVANDO: hay que pesar Z por P(Z|X), no por P(Z).'
    num = sum(p_y_dado_xz(1, x, z) * p_x_dado_z(x, z) * p_z(z) for z in (0, 1))
    den = sum(                       p_x_dado_z(x, z) * p_z(z) for z in (0, 1))
    return num / den

obs1 = p_observando_Y1_dado_X(1)
obs0 = p_observando_Y1_dado_X(0)
print(f"P(Y=1 | X=1)  observando = {obs1:.3f}")
print(f"P(Y=1 | X=0)  observando = {obs0:.3f}")
print(f"Efecto INGENUO (observado) = {obs1 - obs0:+.3f}")

El efecto observado sale **muy por encima** del +0,20 real. ¿Por qué? Porque entre los que reciben X
hay muchos más Z (mayores), y ser Z ya sube Y por su cuenta. Parte de la mejora que le atribuimos a X
es en realidad mérito de Z. Estamos comparando dos grupos que **no** son comparables: ese es el sesgo
del confusor.

### Intervenir: $P(Y\mid \text{do}(X))$

Observar no es intervenir. El operador **do** de Pearl pregunta otra cosa: *¿qué pasaría si
**forzáramos** a todo el mundo a recibir X?* Al fijar X por decreto, **rompemos** la flecha `Z -> X`
(ya no es Z quien decide quién recibe X, lo decidimos nosotros). Eso se llama **mutilar el grafo**:

```
   antes:   Z -> X -> Y , con Z -> Y        despues de do(X):   Z    X=fijo -> Y , con Z -> Y
                                                                  \________________^
```

Z ya no apunta a X. Para calcular $P(Y\mid \text{do}(X=x))$ pesamos por $P(Z)$ **de la población
entera**, no por $P(Z\mid X)$.

In [ ]:
def p_do_Y1(x):
    'P(Y=1 | do(X=x)): grafo mutilado, Z pesa por P(Z), no por P(Z|X).'
    return sum(p_y_dado_xz(1, x, z) * p_z(z) for z in (0, 1))

do1 = p_do_Y1(1)
do0 = p_do_Y1(0)
print(f"P(Y=1 | do(X=1)) = {do1:.3f}")
print(f"P(Y=1 | do(X=0)) = {do0:.3f}")
print(f"Efecto CAUSAL (intervenido) = {do1 - do0:+.3f}")
print()
print(f"Observar dice  {obs1 - obs0:+.3f}  (inflado por el confusor)")
print(f"Intervenir dice {do1 - do0:+.3f}  (el efecto real)")

Ahí está la lección entera del cuaderno en dos números: **observar dice +0,44; intervenir dice +0,20**.
El confusor más que duplica el efecto aparente. Si decidiéramos con el dato observacional, le
atribuiríamos a X el doble de lo que de verdad hace. `P(Y|X)` y `P(Y|do(X))` son cantidades
**distintas**, y confundirlas es el error más caro de la analítica de datos.

## 4. La fórmula de ajuste por puerta trasera

¿Hace falta intervenir de verdad —un experimento— para conocer el efecto causal? No siempre. Si
hemos **medido el confusor** Z, podemos recuperar el efecto de la intervención **a partir de datos de
observación**, con la **fórmula de ajuste** (de la puerta trasera):

$$P(Y \mid \text{do}(X=x)) = \sum_z P(Y \mid X=x, Z=z)\,P(Z=z)$$

La idea: dentro de cada estrato de Z, X **sí** es comparable (el confusor está fijo), así que medimos el
efecto estrato a estrato y luego promediamos con el peso real de cada estrato, $P(Z)$. Comprobamos
numéricamente que **coincide exactamente** con el `do` que calculamos mutilando el grafo.

In [ ]:
def p_ajuste_Y1(x):
    'Formula de ajuste por puerta trasera: suma_z P(Y=1|X,Z) P(Z).'
    return sum(p_y_dado_xz(1, x, z) * p_z(z) for z in (0, 1))

aj1 = p_ajuste_Y1(1)
aj0 = p_ajuste_Y1(0)
print(f"Ajuste  P(Y=1 | X=1) = {aj1:.3f}   ||  do  = {do1:.3f}")
print(f"Ajuste  P(Y=1 | X=0) = {aj0:.3f}   ||  do  = {do0:.3f}")
print(f"Efecto ajustado = {aj1 - aj0:+.3f}   ||  efecto do = {do1 - do0:+.3f}")
print()
print("Coinciden:", np.isclose(aj1, do1) and np.isclose(aj0, do0))
print("El ajuste recupera el efecto causal SIN hacer el experimento, usando solo datos observados.")

Esto es enorme: con el confusor medido y la fórmula correcta, unos datos de pura observación nos dan el
mismo número que daría intervenir. No siempre es posible (hay que haber medido **todos** los confusores
y que no haya puertas traseras abiertas), pero cuando se cumple, es la diferencia entre adivinar y
saber. Herramientas de producción como **DoWhy** o **pgmpy** automatizan justo esta identificación y
este ajuste; aquí lo has hecho a mano para ver que no hay magia, sólo una suma bien pesada.

### El efecto ingenuo frente al efecto ajustado, en una imagen

Tres barras lo resumen: lo que dice mirar los datos sin pensar (ingenuo), lo que dice intervenir (`do`)
y lo que dice la fórmula de ajuste. Las dos últimas son la verdad; coinciden.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
etiquetas = ["ingenuo\nP(Y|X)", "intervenido\nP(Y|do X)", "ajustado\n(puerta trasera)"]
valores = [(obs1 - obs0), (do1 - do0), (aj1 - aj0)]
colores = ["#c0392b", "#27632a", "#27632a"]
barras = ax.bar(etiquetas, valores, color=colores, width=0.6)
ax.axhline(0.20, ls="--", color="gray", lw=1)
ax.text(2.45, 0.205, "efecto real +0.20", ha="right", va="bottom", fontsize=9, color="gray")
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width()/2, v + 0.008, f"{v:+.2f}", ha="center", fontsize=11)
ax.set_ylabel("efecto estimado de X sobre Y")
ax.set_ylim(0, 0.5)
ax.set_title("El confusor infla el efecto: ingenuo vs ajustado/intervenido")
plt.tight_layout(); plt.show()

## 5. El colisionador: condicionar inventa correlaciones

El confusor crea correlación de la nada. Su gemelo malvado, el **colisionador**, hace lo contrario y es
más traicionero: dos causas **independientes** que comparten un **efecto** común. Mientras no toques el
efecto, las causas siguen independientes. Pero en cuanto **condicionas** sobre el efecto (lo fijas, lo
filtras), las dos causas se **acoplan**: aparece una correlación que **no existía**.

El ejemplo clásico: el **talento** y la **suerte** de un proyecto son independientes, pero ambos ayudan
a que te **financien**. Entre los proyectos financiados, talento y suerte salen **correlacionados
negativamente**: si te financiaron a pesar de poco talento, casi seguro tuviste mucha suerte. No es que
una cosa cause la otra; es que has mirado sólo a los seleccionados.

```
   Talento     Suerte        (independientes)
        \       /
         v     v
        Financiacion          <- el COLISIONADOR
```

In [ ]:
n = 4000
talento = rng.normal(0, 1, n)
suerte  = rng.normal(0, 1, n)         # independiente del talento, por construccion
puntuacion = talento + suerte + rng.normal(0, 0.3, n)
financiado = puntuacion > np.quantile(puntuacion, 0.80)   # se financia el 20% con mas puntos

r_todos = corr(talento, suerte)
r_finan = corr(talento[financiado], suerte[financiado])
print(f"corr(talento, suerte) en TODOS los proyectos     = {r_todos:+.3f}  (independientes)")
print(f"corr(talento, suerte) SOLO en los financiados     = {r_finan:+.3f}  (correlacion inventada)")

En la población entera no hay relación; entre los financiados aparece una correlación **negativa**. La
moraleja es la inversa de la del confusor: **no se debe condicionar sobre un colisionador** (ni sobre
sus descendientes). Filtrar por el efecto común introduce el sesgo, no lo quita. Es el mismo *explicar
de más* que viste con la red bayesiana, ahora en lenguaje causal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0), sharex=True, sharey=True)
axes[0].scatter(talento, suerte, s=6, color="#999999", alpha=0.5)
axes[0].set_title(f"Todos los proyectos\ncorr = {r_todos:+.2f}")
axes[1].scatter(talento[~financiado], suerte[~financiado], s=6, color="#dddddd", alpha=0.4, label="no financiado")
axes[1].scatter(talento[financiado], suerte[financiado], s=10, color="#c0392b", alpha=0.7, label="financiado")
axes[1].set_title(f"Solo financiados (en rojo)\ncorr = {r_finan:+.2f}")
for ax in axes:
    ax.set_xlabel("talento"); ax.set_ylabel("suerte")
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()
print("Condicionar sobre el colisionador (financiacion) acopla dos causas que eran independientes.")

## 6. Pruébalo tú

1. **Confusor más fuerte:** sube `P_X1_dado_Z` a `{1: 0.95, 0: 0.05}` (Z decide casi del todo quién
   recibe X) y recalcula el efecto ingenuo. ¿Se infla más o menos el sesgo? El efecto `do` y el
   ajustado, ¿cambian?
2. **Sin confusor:** pon `P_X1_dado_Z = {1: 0.5, 0: 0.5}` (X ya no depende de Z). Comprueba que ahora
   observar e intervenir **coinciden**: sin puerta trasera, $P(Y\mid X) = P(Y\mid \text{do}(X))$.
3. **Simpson al revés:** cambia los datos de cálculos renales para que sea **B** quien acapare los casos
   graves. ¿Quién gana ahora en el total?
4. **Colisionador más selectivo:** financia sólo el 5 % mejor (`np.quantile(..., 0.95)`). ¿La
   correlación inventada se hace más fuerte o más débil?

## 7. Errores comunes

- **«Están correlacionados, luego uno causa al otro.»** El error fundacional. Puede haber un confusor
  detrás moviendo los hilos de los dos.
- **Confiar en la media agregada.** Simpson enseña que el total puede decir lo contrario que cada
  subgrupo. Antes de promediar, pregunta qué confusor estás mezclando.
- **Tratar $P(Y\mid X)$ como si fuera $P(Y\mid \text{do}(X))$.** Observar mide asociación; intervenir
  mide causa. Sólo coinciden si no hay puertas traseras abiertas.
- **Ajustar por todo «por si acaso».** Controlar variables a ciegas es peligroso: si una de ellas es un
  **colisionador**, ajustar por ella **crea** sesgo en lugar de quitarlo. Hay que saber qué es cada nodo.
- **Creer que el ajuste siempre vale.** La fórmula de la puerta trasera sólo recupera el efecto si has
  medido **todos** los confusores. Un confusor no observado y el número vuelve a mentir.

## 8. Qué te llevas

- **Correlación no es causa.** Un **confusor** (causa común) basta para que dos cosas vayan de la mano
  sin influirse: los helados y los ahogamientos suben juntos por el calor, y al controlar la temperatura
  la relación se desploma.
- La **paradoja de Simpson** es el aviso más serio: con datos reales, el tratamiento A gana en cálculos
  pequeños (93,1 % frente a 86,7 %) y en grandes (73,0 % frente a 68,8 %), pero **pierde en el total**
  (78,0 % frente a 82,6 %). La mezcla invierte la conclusión.
- El **operador do** separa observar de intervenir. En nuestro modelo, **observar dice +0,44 e intervenir
  dice +0,20**: el confusor más que duplica el efecto aparente.
- La **fórmula de ajuste por puerta trasera** recupera ese +0,20 real a partir de datos de observación,
  sumando el efecto dentro de cada estrato del confusor pesado por $P(Z)$. Coincide al decimal con el `do`.
- Un **colisionador** hace lo contrario del confusor: condicionar sobre un efecto común **inventa** una
  correlación entre causas que eran independientes. Por eso no se ajusta por todo a ciegas.

**Para seguir:** en producción esto se modela con **DoWhy**, **pgmpy** o **CausalNex**, que identifican
el conjunto de ajuste y estiman el efecto por ti; pero la gramática es la que has visto aquí. La causa
es la pregunta que un modelo, por grande que sea, no contesta sólo mirando correlaciones: hay que decirle
qué provoca qué.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*